# Download Data for GraphCast

Downloads data from the ERA5 database and prepares them to be suitable input for GraphCast.
As parameters the desired date and times can be chosen.
Addiotionally can be chosen between 13 and 37 pressure levels and a resolution of 0.25 or 1.0 degrees to match different GraphCast models.

In [1]:
import os
import cdsapi
import xarray as xr
import numpy as np
import zipfile

In [2]:
def prepare_graphcast_input(date: str, times: list[str], levels: int = 13, resolution: float = 0.25):
    assert levels in [13, 37], "Only 13 or 37 pressure levels supported."
    assert resolution in [0.25, 1.0], "Only 0.25 or 1.0 degree resolution supported."

    # Set pressure levels
    levels_13 = ['50', '100', '150', '200', '250', '300', '400', '500',
                 '600', '700', '850', '925', '1000']
    levels_37 = [str(l) for l in (
        [1, 2, 3, 5, 7, 10, 20, 30, 50, 70, 100, 125, 150, 175, 200, 225, 250,
         300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 775, 800, 825, 850,
         875, 900, 925, 950, 975, 1000]
    )]
    pressure_levels = levels_13 if levels == 13 else levels_37

    year, month, day = date.split("-")
    time_tag = "-".join(t.replace(":", "") for t in times)
    res_tag = f"res{int(resolution * 100)}"
    tag = f"{date}-{levels}lev-{time_tag}-{res_tag}"
    folder = f"data/input_data/{tag}"
    os.makedirs(folder, exist_ok=True)

    c = cdsapi.Client()

    print("Downloading pressure-level data...")
    c.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                'temperature', 'u_component_of_wind', 'v_component_of_wind',
                'geopotential', 'vertical_velocity', 'specific_humidity'
            ],
            'pressure_level': pressure_levels,
            'year': year, 'month': month, 'day': day,
            'time': times,
            'grid': [resolution, resolution],
            'format': 'netcdf',
        },
        f"{folder}/era5_pressure.nc"
    )

    print("Downloading surface-level data...")
    surface_path = f"{folder}/era5_surface.zip"
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                '2m_temperature', '10m_u_component_of_wind',
                '10m_v_component_of_wind', 'mean_sea_level_pressure',
                'total_precipitation'
            ],
            'year': year, 'month': month, 'day': day,
            'time': times,
            'grid': [resolution, resolution],
            'format': 'netcdf',
        },
        surface_path
    )

    print("Extracting surface ZIP...")
    extract_dir = f"{folder}/surface_extracted"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(surface_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Load pressure
    pressure_ds = xr.open_dataset(f"{folder}/era5_pressure.nc", engine="netcdf4")
    pressure_ds = pressure_ds.rename({
        "valid_time": "time", "latitude": "lat", "longitude": "lon", "pressure_level": "level"
    }).expand_dims("batch")

    # Load and merge surface
    instant_ds = xr.open_dataset(f"{extract_dir}/data_stream-oper_stepType-instant.nc", engine="netcdf4")
    accum_ds = xr.open_dataset(f"{extract_dir}/data_stream-oper_stepType-accum.nc", engine="netcdf4")
    surface_ds = xr.merge([instant_ds, accum_ds])
    surface_ds = surface_ds.rename({
        "valid_time": "time", "latitude": "lat", "longitude": "lon"
    }).expand_dims("batch")
    surface_ds["time"] = pressure_ds["time"]

    combined_ds = xr.merge([pressure_ds, surface_ds])
    combined_ds = combined_ds.rename({
        "t": "temperature",
        "u": "u_component_of_wind",
        "v": "v_component_of_wind",
        "z": "geopotential",
        "w": "vertical_velocity",
        "q": "specific_humidity",
        "t2m": "2m_temperature",
        "u10": "10m_u_component_of_wind",
        "v10": "10m_v_component_of_wind",
        "msl": "mean_sea_level_pressure",
        "tp": "total_precipitation_6hr"
    }).drop_vars(["number", "expver"], errors="ignore")

    # Add datetime coordinate
    time_coord = combined_ds["time"]
    batch_size = combined_ds.sizes["batch"]
    datetime_broadcast = xr.DataArray(
        np.broadcast_to(time_coord.values, (batch_size, len(time_coord))),
        dims=("batch", "time")
    )
    combined_ds = combined_ds.assign_coords(datetime=datetime_broadcast)

    # geopotential_at_surface (static from 1000 hPa)
    combined_ds["geopotential_at_surface"] = combined_ds["geopotential"].sel(level=1000).isel(time=0)

    print("Downloading land-sea mask...")
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': ['land_sea_mask'],
            'year': year, 'month': month, 'day': day,
            'time': ['00:00'],
            'grid': [resolution, resolution],
            'format': 'netcdf',
        },
        f"{folder}/land_sea_mask.nc"
    )
    lsm_ds = xr.open_dataset(f"{folder}/land_sea_mask.nc")
    lsm_ds = lsm_ds.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
    lsm = lsm_ds["lsm"].isel(time=0).squeeze()
    combined_ds["land_sea_mask"] = lsm

    # Ensure 'geopotential_at_surface' has no time dim
    if "time" in combined_ds["geopotential_at_surface"].dims:
        combined_ds["geopotential_at_surface"] = combined_ds["geopotential_at_surface"].isel(time=0)

    # Save with clear filename
    output_file = f"{folder}/graphcast_ready_input_{tag}.nc"
    combined_ds.to_netcdf(output_file)
    print(f"Saved: {output_file}")


In [4]:
# GraphCast_small

prepare_graphcast_input("2022-02-18", ["09:00", "15:00", "21:00"], levels=13, resolution=1.0)

2025-05-08 10:17:10,278 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-05-08 10:17:10,279 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


2025-05-08 10:17:10,533 INFO Request ID is 15ecc9b9-5cff-4cd4-a03c-0450632f7085
2025-05-08 10:17:10,621 INFO status has been updated to accepted
2025-05-08 10:17:24,549 INFO status has been updated to running
2025-05-08 10:17:43,737 INFO status has been updated to successful


4326cf63ad330ec5c79a3514dbab420.nc:   0%|          | 0.00/28.5M [00:00<?, ?B/s]

2025-05-08 10:17:46,568 INFO Request ID is c619383a-e37b-4580-94a3-1323ab24c9ae
2025-05-08 10:17:46,647 INFO status has been updated to accepted
2025-05-08 10:18:08,115 INFO status has been updated to running
2025-05-08 10:18:20,178 INFO status has been updated to successful


f62f151bde81d809fd81d678cd2a7c2.zip:   0%|          | 0.00/1.76M [00:00<?, ?B/s]

Extracting surface ZIP...


2025-05-08 10:18:22,327 INFO Request ID is ec950716-84f4-485a-bef2-9febab6707f8
2025-05-08 10:18:22,403 INFO status has been updated to accepted
2025-05-08 10:18:36,393 INFO status has been updated to running
2025-05-08 10:18:44,077 INFO status has been updated to successful


115c90d00365360e2e1d801f56f491e9.nc:   0%|          | 0.00/83.8k [00:00<?, ?B/s]

Saved: data/input_data/2022-02-18-13lev-0900-1500-2100-res100/graphcast_ready_input_2022-02-18-13lev-0900-1500-2100-res100.nc


In [3]:
# GraphCast_operational

prepare_graphcast_input("2022-02-18", ["09:00", "15:00", "21:00"], levels=13, resolution=0.25)

2025-05-08 11:38:27,391 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-05-08 11:38:27,392 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


2025-05-08 11:38:27,789 INFO Request ID is a3570f08-2ee4-4813-af99-aaada810f8a5
2025-05-08 11:38:27,895 INFO status has been updated to accepted
2025-05-08 11:38:49,194 INFO status has been updated to running
2025-05-08 11:40:22,118 INFO status has been updated to successful


ea3a84f22910cfa4b7d707e4a327057.nc:   0%|          | 0.00/391M [00:00<?, ?B/s]

2025-05-08 11:41:02,989 INFO Request ID is 95c248e4-e825-4fee-ad75-b462cee563e2
2025-05-08 11:41:03,116 INFO status has been updated to accepted
2025-05-08 11:41:16,981 INFO status has been updated to running
2025-05-08 11:41:53,284 INFO status has been updated to successful


8dcfd36851f1ca041242f398ce69ed3e.zip:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

Extracting surface ZIP...


2025-05-08 11:41:59,553 INFO Request ID is 5082cd91-1a79-4329-846e-422d0b677312
2025-05-08 11:41:59,613 INFO status has been updated to accepted
2025-05-08 11:42:49,545 INFO status has been updated to successful


9655d79da5d417fa3b161d892fc88eae.nc:   0%|          | 0.00/762k [00:00<?, ?B/s]

Saved: data/input_data/2022-02-18-13lev-0900-1500-2100-res25/graphcast_ready_input_2022-02-18-13lev-0900-1500-2100-res25.nc


In [4]:
# GraphCast

prepare_graphcast_input("2022-02-18", ["09:00", "15:00", "21:00"], levels=37, resolution=0.25)

2025-05-08 11:43:54,870 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-05-08 11:43:54,871 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


2025-05-08 11:43:55,573 INFO Request ID is e0a6a909-777e-4aa8-928e-05665f3be5f1
2025-05-08 11:43:55,796 INFO status has been updated to accepted
2025-05-08 11:45:11,763 INFO status has been updated to running
2025-05-08 11:52:15,686 INFO status has been updated to successful


74fce878e9fc91838a7a03c3b5345a79.nc:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

2025-05-08 11:53:37,890 INFO Request ID is 9d20af9e-50d1-477c-ac4c-5bac9a0cd9f5
2025-05-08 11:53:37,978 INFO status has been updated to accepted
2025-05-08 11:53:46,587 INFO status has been updated to running
2025-05-08 11:53:51,719 INFO status has been updated to successful


8dcfd36851f1ca041242f398ce69ed3e.zip:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

Extracting surface ZIP...


2025-05-08 11:54:02,910 INFO Request ID is cc3856a4-af17-4f3d-b4e2-2203ec22bf37
2025-05-08 11:54:03,013 INFO status has been updated to accepted
2025-05-08 11:54:35,822 INFO status has been updated to successful


9655d79da5d417fa3b161d892fc88eae.nc:   0%|          | 0.00/762k [00:00<?, ?B/s]

Saved: data/input_data/2022-02-18-37lev-0900-1500-2100-res25/graphcast_ready_input_2022-02-18-37lev-0900-1500-2100-res25.nc
